# 🌿 Ambient Wildlife Poaching Guardian (Championship Capstone Submission)

This notebook demonstrates the **Ambient Wildlife Poaching Guardian**, a resilient anti-poaching operations fabric built using **Google Agent Development Kit (ADK 2.0)**.

## 📖 Project Overview
The project models a **Common Operational Picture (COP)** for conservation. Rather than just alert on audio, it aggregates reserve context (weather, patrols, populations) using MCP tools, applies safety policy guardrails, pauses for Human-in-the-Loop confirmations, coordinates drone/ranger dispatch, and tracks quality flywheel metrics. To run at **$0 cloud cost**, it falls back to a local rules engine when cloud API keys are absent.

## 📐 Agent Graph Topology

```
[Acoustic Sensor Event]
  -> Sensor Intake Agent (Standardizes CloudEvent envelope)
  -> Acoustic Triage Agent (Pre-filters quiet eco sounds)
  -> Context Fusion Agent (MCP Weather, SMART Patrol coordinates check)
  -> Threat Assessor Agent (SafeLlmAgent structured assessment)
  -> Policy & Safety Agent (Deterministic safety overlays)
  -> Human Confirmation Check (HITL Pause & Rehydrate)
  -> Dispatcher Agent (Coordinates Drone & Ranger assets)
  -> After Action Agent (Compiles Markdown telemetry report)
```

## 🚀 Code Execution Demo

Let's execute a simulation run locally in this notebook. We will trigger a **gunshot in Sector 4** where high winds (>40 km/h) are forecasted. We will observe the Context Fusion, the Policy and Safety override redirecting the drone launch to a ranger patrol, the HITL pause, and the rehydrated completion.

In [ ]:
import json
import os
import sys

# Ensure project path is resolved
sys.path.insert(0, os.path.abspath("."))

# Force SafeLlmAgent local mock mode
os.environ["GEMINI_API_KEY"] = ""

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from app.agent import root_agent

# Initialize local ADK runner
session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent, session_service=session_service, app_name="wildlife_guardian"
)

# Telemetry: Gunshot in Sector 4 (high wind sector)
payload = {
    "sensor_id": "sensor-aud-01",
    "location": "Sector 4",
    "decibel_level": 110.0,
    "acoustic_signature": "Possible Gunshot",
    "timestamp": "2026-06-22T08:00:00Z",
}

print("📥 Ingesting Telemetry into Sensor Intake...")
session = session_service.create_session_sync(
    user_id="ranger_station", app_name="wildlife_guardian"
)
message = types.Content(
    role="user", parts=[types.Part.from_text(text=json.dumps(payload))]
)

# Run the multi-agent graph
events = list(
    runner.run(user_id="ranger_station", session_id=session.id, new_message=message)
)

print("\n⚡ Active Agent execution trace:")
interrupted = False
interrupt_id = None
for e in events:
    if e.content and e.content.parts:
        for part in e.content.parts:
            if part.function_call and part.function_call.name == "adk_request_input":
                interrupted = True
                interrupt_id = part.function_call.id
                print(
                    f"\n⏸️  Workflow Paused (HITL Wait):\n{part.function_call.args.get('message')}"
                )
    if e.output:
        # Print internal updates
        pass

# Resume the paused session simulating Ranger Dispatch approval
if interrupted and interrupt_id:
    print("\n💂 Simulating Ranger Dispatch Approval...")
    resume_message = types.Content(
        role="user",
        parts=[
            types.Part(
                function_response=types.FunctionResponse(
                    id=interrupt_id, response={"response": "approve"}
                )
            )
        ],
    )
    resume_events = list(
        runner.run(
            user_id="ranger_station", session_id=session.id, new_message=resume_message
        )
    )
    for re in resume_events:
        if re.output:
            print(
                f"\n📊 Final After-Action Mission Report:\n{json.dumps(re.output, indent=2)}"
            )